## Решение заданий с помощью Spark.SQL

Установим Spark подключение и проверим его.

In [1]:
# import pyspark as spark

#

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Hadoop_Assignment_VSCode") \
    .config("spark.local.dir", "/home/s.yudin/spark-temp") \
    .enableHiveSupport() \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/15 16:32:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/15 16:32:51 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/06/15 16:32:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
spark.sql("USE s_yudin")

26/06/15 16:32:59 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [3]:
#test
spark.sql("""
    SELECT * FROM dim_cat LIMIT 5
""").show()

+------+--------------------+
|cat_id|       category_name|
+------+--------------------+
|     1|                  $1|
|     2|                 $10|
|     3|          $10, Eagle|
|     4|                $100|
|     5|$2.50, Quarter Eagle|
+------+--------------------+



Создаём внешнюю таблицу поверх Parquet-файлов:

In [6]:
# Путь к данным (из логов ранее)
parquet_path = "/data/raw/yndx_metrica/parquet/date=2026"

# Прочитаем один файл, чтобы увидеть схему (выполните один раз)
df_sample = spark.read.parquet(parquet_path)
df_sample.printSchema()


root
 |-- date: string (nullable = true)
 |-- dt: string (nullable = true)
 |-- visitID: string (nullable = true)
 |-- isNewUser: long (nullable = true)
 |-- startURL: string (nullable = true)
 |-- endURL: string (nullable = true)
 |-- pageViews: long (nullable = true)
 |-- visitDuration: long (nullable = true)
 |-- regionCountry: string (nullable = true)
 |-- regionCity: string (nullable = true)
 |-- clientID: string (nullable = true)
 |-- ipAddress: string (nullable = true)
 |-- clientTimeZone: long (nullable = true)
 |-- deviceCategory: long (nullable = true)
 |-- mobilePhone: string (nullable = true)
 |-- mobilePhoneModel: string (nullable = true)
 |-- operatingSystem: string (nullable = true)
 |-- browser: string (nullable = true)



In [ ]:
# Создаём внешнюю таблицу
spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS yandex_metrica (
        dt STRING,
        visitID STRING,
        isNewUser BIGINT,
        startURL STRING,
        endURL STRING,
        pageViews BIGINT,
        visitDuration BIGINT,
        regionCountry STRING,
        regionCity STRING,
        clientID STRING,
        ipAddress STRING,
        clientTimeZone BIGINT,
        deviceCategory BIGINT,
        mobilePhone STRING,
        mobilePhoneModel STRING,
        operatingSystem STRING,
        browser STRING
    )
    PARTITIONED BY (date STRING)
    STORED AS PARQUET
    LOCATION '/data/raw/yndx_metrica/parquet'
""")


26/06/15 12:53:12 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/06/15 12:53:13 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


DataFrame[]

In [8]:
# Регистрируем существующие партиции (например, date=2026)
spark.sql("MSCK REPAIR TABLE yandex_metrica")

DataFrame[]

### Блок 1: Исследование и базовый анализ

Задача 1. Проверка структуры и форматов таблицы yandex_metrica

---

In [9]:
# Подробная информация о таблице
spark.sql("DESC FORMATTED yandex_metrica").show(100, truncate=False)

+----------------------------+--------------------------------------------------------------+-------+
|col_name                    |data_type                                                     |comment|
+----------------------------+--------------------------------------------------------------+-------+
|dt                          |string                                                        |NULL   |
|visitID                     |string                                                        |NULL   |
|isNewUser                   |bigint                                                        |NULL   |
|startURL                    |string                                                        |NULL   |
|endURL                      |string                                                        |NULL   |
|pageViews                   |bigint                                                        |NULL   |
|visitDuration               |bigint                                              

In [10]:
# Формат хранения и партиции
spark.sql("SHOW CREATE TABLE yandex_metrica").show(truncate=False)


+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|createtab_stmt                                                                                                                                                                                                                                                                                                                                                                                                    

In [11]:
spark.sql("SHOW PARTITIONS yandex_metrica").show()

+---------+
|partition|
+---------+
|date=2022|
|date=2023|
|date=2024|
|date=2025|
|date=2026|
+---------+



Задача 2. Анализ полноты данных (пустые города)

---

In [15]:
spark.sql("""
    SELECT 
        COUNT(*) AS total_records,
        SUM(CASE WHEN regionCity IS NULL OR regionCity = '' THEN 1 ELSE 0 END) AS empty_city_count,
        ROUND(SUM(CASE WHEN regionCity IS NULL OR regionCity = '' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS empty_city_percent
    FROM yandex_metrica
""").show()

26/06/15 13:02:06 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


+-------------+----------------+------------------+
|total_records|empty_city_count|empty_city_percent|
+-------------+----------------+------------------+
|      2832442|          541365|             19.11|
+-------------+----------------+------------------+



### Блок 2: Аналитика Яндекс.Метрики

Задача 3. ТОП-10 популярных браузеров

---

In [ ]:
spark.sql("""
    SELECT 
        browser,
        COUNT(*) AS visits
    FROM yandex_metrica
    WHERE browser IS NOT NULL AND browser != ''
    GROUP BY browser
    ORDER BY visits DESC
    LIMIT 10
""").show()

26/06/15 12:58:41 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


+----------------+------+
|         browser|visits|
+----------------+------+
|          chrome|730582|
|   safari_mobile|664030|
|    chromemobile|606580|
|  yandex_browser|247926|
| android_browser|136677|
|          safari| 69693|
|            edge| 68632|
|samsung_internet| 57725|
|            miui| 48478|
|           opera| 46720|
+----------------+------+



Задача 4. Среднее количество просмотров страниц по городам (>3)

---

In [ ]:
spark.sql("""
    SELECT 
        regionCity,
        AVG(pageViews) AS avg_pageviews
    FROM yandex_metrica
    WHERE regionCity IS NOT NULL AND regionCity != ''
    GROUP BY regionCity
    HAVING AVG(pageViews) > 3
    ORDER BY avg_pageviews DESC
""").show()

26/06/15 12:59:43 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


+--------------------+------------------+
|          regioncity|     avg_pageviews|
+--------------------+------------------+
|Bietigheim-Bissingen|              17.0|
|           Maidstone|              14.0|
|               Hanau|13.428571428571429|
|             Pistoia|              13.0|
|        Mount Juliet|              13.0|
|           Siechnice|              12.0|
|               Payas|              11.0|
|             Skoczow|              11.0|
|                Tías|10.333333333333334|
|            Wolsztyn|              10.0|
|           Pirmasens|              10.0|
|                Deva|               9.0|
|Upper Darby Township|               9.0|
|           Podbořany|               9.0|
|             Chengdu| 8.636363636363637|
|               Vaasa|               8.0|
|            Borlange|               8.0|
|              Granby|               8.0|
|        Markkleeberg|               8.0|
|          Neumünster|               7.5|
+--------------------+------------

### Блок 3: Аналитика eBay

Задача 5. Ценовой анализ по категориям (price > 0)

---

In [16]:
spark.sql("""
    SELECT 
        category_name,
        MIN(price) AS min_price,
        MAX(price) AS max_price,
        AVG(price) AS avg_price
    FROM ebay_listings_optimized
    WHERE price > 0
    GROUP BY category_name
    ORDER BY category_name
""").show()

26/06/15 13:03:51 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


+--------------------+---------+---------+------------------+
|       category_name|min_price|max_price|         avg_price|
+--------------------+---------+---------+------------------+
|                NULL|      5.0|   238.06|50.944810810810786|
|                  $1|     2.99|   990.76|147.39661290322582|
|                 $10|    88.88|   977.98|          280.8275|
|          $10, Eagle|     20.0|    34.99| 31.97826086956522|
|                $100|    10.86|   604.72|115.11142857142859|
|$2.50, Quarter Eagle|    57.66|    58.28|             57.97|
|   1 Pfennig ab 1950|    23.12|    23.12|             23.12|
|10-Euro-Gedenkmünzen|    21.79|    21.79|             21.79|
|                1000|     6.66|    158.8| 61.96999999999999|
|              100005|     15.0|    54.38|           30.0925|
|              100011|    71.99|    71.99|             71.99|
|              100012|    27.81|    37.93|             32.87|
|              100058|     15.4|   207.89| 90.42894736842106|
|       

Задача 6. Рейтинг стран по количеству уникальных продавцов (ТОП-5)

---

In [17]:
spark.sql("""
    SELECT 
        location_country,
        COUNT(DISTINCT seller_name) AS unique_sellers
    FROM ebay_listings_optimized
    WHERE location_country IS NOT NULL AND location_country != ''
    GROUP BY location_country
    ORDER BY unique_sellers DESC
    LIMIT 5
""").show()

26/06/15 13:05:07 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist


+----------------+--------------+
|location_country|unique_sellers|
+----------------+--------------+
|              US|        673517|
|              GB|         31452|
|              CN|         20695|
|              CA|         11399|
|              AU|         10042|
+----------------+--------------+



### Блок 4: Продвинутые задачи (Data Quality)

Задача 7. Поиск аномалий в визитах (visitduration > 3600 и pageviews = 1)

---

In [4]:
spark.sql("""
    SELECT *
    FROM yandex_metrica
    WHERE visitDuration > 3600 AND pageViews = 1
    LIMIT 10
""").show()

+-------------------+-------------------+---------+--------------------+--------------------+---------+-------------+-------------+-------------+--------------------+--------------------+--------------+--------------+-----------+----------------+---------------+-------------+----+
|                 dt|            visitID|isNewUser|            startURL|              endURL|pageViews|visitDuration|regionCountry|   regionCity|            clientID|           ipAddress|clientTimeZone|deviceCategory|mobilePhone|mobilePhoneModel|operatingSystem|      browser|date|
+-------------------+-------------------+---------+--------------------+--------------------+---------+-------------+-------------+-------------+--------------------+--------------------+--------------+--------------+-----------+----------------+---------------+-------------+----+
|2025-05-30 18:02:12|2958592012760121677|        0|https://astondevs...|https://astondevs...|        1|         7374|       Russia|         NULL|1.7483511

Задача 8. Количество визитов по дням и поиск «провалов»

---

In [9]:
# Базовый подсчёт по дням из партиции
spark.sql("""
    SELECT 
        dt::date,
        COUNT(*) AS visits
    FROM yandex_metrica
    GROUP BY dt::date
    ORDER BY dt::date
""").show()


+----------+------+
|        dt|visits|
+----------+------+
|2022-09-08|    13|
|2022-09-09|   252|
|2022-09-10|   299|
|2022-09-11|   446|
|2022-09-12|   495|
|2022-09-13|   798|
|2022-09-14|   769|
|2022-09-15|   457|
|2022-09-16|   634|
|2022-09-17|   568|
|2022-09-18|   650|
|2022-09-19|   930|
|2022-09-20|   522|
|2022-09-21|   473|
|2022-09-22|   510|
|2022-09-23|   405|
|2022-09-24|    96|
|2022-09-25|   161|
|2022-09-26|   441|
|2022-09-27|   312|
+----------+------+
only showing top 20 rows


In [12]:
# Поиск провалов в посещениях


threshold = 50          # порог, ниже которого день считается "провалом"

spark.sql(f"""
    WITH daily_visits AS (
        SELECT 
            dt::date AS day,
            COUNT(*) AS visits
        FROM yandex_metrica
        GROUP BY dt::date
    )
    SELECT 
          day,
          COALESCE(visits, 0) AS visits
    FROM daily_visits
    WHERE COALESCE(visits, 0) < {threshold}
""").show()

+----------+------+
|       day|visits|
+----------+------+
|2022-09-08|    13|
+----------+------+

